# From data to geometry

Run each cell in order to build a 3D bar chart inside Blender.

In [ ]:
import bpy
import numpy as np

print("Blender", bpy.app.version_string)

## Data

In [ ]:
months = np.array(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
rainfall = np.array(
    [62, 47, 54, 41, 78, 99, 121, 110, 84, 67, 58, 70], dtype=float
)

## Build bars

In [ ]:
COLLECTION = "DataBars"
col = bpy.data.collections.new(COLLECTION)
bpy.context.scene.collection.children.link(col)

heights = rainfall / 20.0
bars = []
for i, h in enumerate(heights):
    bpy.ops.mesh.primitive_cube_add(size=1.0)
    bar = bpy.context.active_object
    bar.name = f"bar_{months[i]}"
    bar.scale = (0.4, 0.4, float(h))
    bar.location = (i - len(heights) / 2.0, 0.0, h / 2.0)
    col.objects.link(bar)
    for c in list(bar.users_collection):
        if c is not col:
            c.objects.unlink(bar)
    bars.append(bar)

## Color by value

In [ ]:
def value_to_color(t):
    return (t, 0.15, 1.0 - t, 1.0)

span = rainfall.max() - rainfall.min()
norm = (rainfall - rainfall.min()) / span
for bar, t in zip(bars, norm):
    mat = bpy.data.materials.new(name=f"mat_{bar.name}")
    if mat.node_tree is None:
        mat.use_nodes = True
    bsdf = mat.node_tree.nodes["Principled BSDF"]
    bsdf.inputs["Base Color"].default_value = value_to_color(float(t))
    bar.data.materials.clear()
    bar.data.materials.append(mat)

## Lights

In [ ]:
lights_col = bpy.data.collections.new("DataLights")
bpy.context.scene.collection.children.link(lights_col)

for bar, h in zip(bars, heights):
    light_data = bpy.data.lights.new(name=f"light_{bar.name}", type='POINT')
    light_data.energy = 30
    light_obj = bpy.data.objects.new(name=f"light_{bar.name}", object_data=light_data)
    # slightly above the bar top, slightly in front (negative Y toward camera)
    light_obj.location = (bar.location.x, -1.5, h + 1.0)
    lights_col.objects.link(light_obj)